# Municipal Bonds Data Extraction Pipeline

This notebook extracts text, 'Use of Proceeds' tables, and 'Jurisdiction' tables from the `US Counties - Muni Bonds Data.pdf` report. It relies on `pdfplumber` to extract text from the PDF, using Regex patterns to safely pull statistics and tabular data.

## 1. Extract District Level Data (Text)

In [ ]:
import pdfplumber
import pandas as pd
import re
import os

filename = "US Counties - Muni Bonds Data.pdf"

if os.path.exists(filename):
    full_path = filename
elif os.path.exists(os.path.join('documents', 'muni', filename)):
    full_path = os.path.join('documents', 'muni', filename)
else:
    raise FileNotFoundError(f"Could not find {filename}")

patterns = {
    # District Identity
    "District_Name": r"Municipal Bonds in (?:the\s+)?(.+?)\s+Congressional District",
    "District_Name_Backup": r"For\s+(?:the\s+)?(.+?)\s+Congressional\s+District\s+that\s+means:",
    
    # Issuers & Borrowers (Space-Optional)
    "Total_Issuers": r"(\d+(?:,\d{3})*)\s*(?:sub[\s\-\u2013\u2014]*state|state\s*and\s*local|local)\s*governments",
    "Small_Borrowers": r"(\d+(?:,\d{3})*)\s*have\s*(?:borrowed\s*)?less\s*than",
    
    # Investment & Savings (Space-Optional)
    "Sub_State_Inv_Value": r"invested\s*(?:at\s*least\s*)?\$(\d[\d,.]*)",
    "Sub_State_Inv_Unit":  r"invested\s*(?:at\s*least\s*)?\$\d[\d,.]*\s*(billion|million)",
    "Total_Inv_Value":     r"total\s*investment\s*is\s*\$(\d[\d,.]*)",
    "Sub_State_Sav_Value": r"saved\s*(?:at\s*least\s*)?(?:an\s*estimated\s*)?\$(\d[\d,.]*)",
    
    # Percentage (Space-Optional Update)
    "Small_Borrowers_Pct": r"(?:represent|are)\s*([\d.]+)%"
}

def extract_paragraph_data(pdf_path):
    data = []
    print(f"Scanning text in: {os.path.basename(pdf_path)}...")
    
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages):
            raw_text = page.extract_text()
            if not raw_text: continue
            
            clean_text = raw_text.replace('\n', ' ')
            record = {"Page": i + 1}
            
            for key, pattern in patterns.items():
                match = re.search(pattern, clean_text, re.IGNORECASE)
                if match:
                    record[key] = match.group(1).strip()
                else:
                    if key not in record: record[key] = None

            if not record["District_Name"] and record.get("District_Name_Backup"):
                record["District_Name"] = record["District_Name_Backup"]

            if record["District_Name"]:
                if "District_Name_Backup" in record: del record["District_Name_Backup"]
                data.append(record)
                
    return pd.DataFrame(data)

df_text = extract_paragraph_data(full_path)
print(f"\nSuccess! Scraped {len(df_text)} districts.")

df_text.to_csv("Muni_Text_Data_Final.csv", index=False)
print("\nSaved to Muni_Text_Data_Final.csv")


## 2. Extract Use of Proceeds Tables

In [ ]:
def clean_money(val_str):
    if not val_str: return 0.0
    clean = re.sub(r'[^\d.]', '', val_str)
    try:
        return float(clean)
    except ValueError:
        return 0.0

def scrape_proceeds_table(pdf_path):
    all_rows = []
    print(f"Scraping 'Use of Proceeds' from: {os.path.basename(pdf_path)}...")
    
    current_district = "Unknown"
    
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages):
            text = page.extract_text()
            if not text: continue
            
            header_match = re.search(r"Municipal Bonds in (?:the\s+)?(.+?)\s+Congressional District", text)
            if header_match:
                current_district = header_match.group(1).strip()
            
            if current_district != "Unknown":
                lines = text.split('\n')
                in_table = False
                
                for line in lines:
                    clean_line = line.strip()
                    if re.search(r"Use\s*of\s*Proceeds", clean_line, re.IGNORECASE):
                        in_table = True
                        continue
                    if in_table and re.search(r"^Total", clean_line, re.IGNORECASE):
                        in_table = False
                        continue
                    
                    if in_table and clean_line:
                        if "Investment" in clean_line and "millions" in clean_line:
                            continue
                        amount_match = re.search(r'(\$?[\d,]+\.\d{2})$', clean_line)
                        if amount_match:
                            amount_str = amount_match.group(1)
                            category = clean_line.replace(amount_str, "").strip()
                            all_rows.append({
                                "Page": i + 1,
                                "District_Name": current_district,
                                "Category": category,
                                "Investment_Amount": clean_money(amount_str)
                            })

    return pd.DataFrame(all_rows)

df_proceeds = scrape_proceeds_table(full_path)
print(f"\nSuccess! Found {len(df_proceeds)} rows.")

df_proceeds.to_csv("Muni_Use_Of_Proceeds_Final.csv", index=False)
print("\nSaved to Muni_Use_Of_Proceeds_Final.csv")


## 3. Extract Jurisdiction Tables

In [ ]:
def scrape_jurisdiction_table(pdf_path):
    all_rows = []
    print(f"Scraping 'Jurisdiction' tables from: {os.path.basename(pdf_path)}...")
    
    current_district = "Unknown"
    in_jurisdiction_table = False
    
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages):
            text = page.extract_text(x_tolerance=1)
            if not text: continue
            
            header_match = re.search(r"Municipal Bonds in (?:the\s+)?(.+?)\s+Congressional District", text)
            if header_match:
                current_district = header_match.group(1).strip()
                in_jurisdiction_table = False 
            
            lines = text.split('\n')
            for line in lines:
                clean_line = line.strip()
                if re.search(r"Jurisdiction", clean_line, re.IGNORECASE) and not in_jurisdiction_table:
                    in_jurisdiction_table = True
                    continue
                if in_jurisdiction_table and re.search(r"^Total", clean_line, re.IGNORECASE):
                    in_jurisdiction_table = False
                    continue
                
                if in_jurisdiction_table and clean_line:
                    if re.search(r"Investment", clean_line, re.IGNORECASE):
                        continue
                    amount_match = re.search(r'(\$?[\d,]+(?:\.\d{2})?)$', clean_line)
                    if amount_match:
                        amount_str = amount_match.group(1)
                        issuer = clean_line.replace(amount_str, "").strip()
                        if current_district != "Unknown":
                            all_rows.append({
                                "Page": i + 1,
                                "District_Name": current_district,
                                "Issuer": issuer,
                                "Investment_Amount": clean_money(amount_str)
                            })

    return pd.DataFrame(all_rows)

df_jurisdiction = scrape_jurisdiction_table(full_path)
print(f"\nSuccess! Found {len(df_jurisdiction)} issuer records.")

df_jurisdiction.to_csv("Muni_Jurisdiction_Final.csv", index=False)
print("\nSaved to Muni_Jurisdiction_Final.csv")
